# F1 data prep: time series → one tabular `Xy` for all 23 F1 units

**Goal.** Turn every F1 unit's panel into rows of the form

> *(unit, asset, origin date, horizon) → feature vector at the origin → target value h business days later*

so any tabular model (ridge, LightGBM, quantile regression, …) can be trained and evaluated on it.
**No model is fit here** — this notebook only produces the dataset and documents it.

Output: `data_prep/f1_Xy.parquet` (long format, one row per unit × asset × origin × horizon),
plus `f1_Xy_predict.parquet` (just the 43 rows that need forecasting) and `f1_columns.json`.

## 0 — Setup

In [2]:
import json, pathlib, tomllib, time
import numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 40)

REPO = pathlib.Path.cwd()
OUT  = REPO / "data_prep"; OUT.mkdir(exist_ok=True)

## 1 — What the 23 F1 units look like

They are not homogeneous. Four panels, two target types, one or two horizons, and two are monthly.

In [3]:
def load_unit(unit_dir: pathlib.Path) -> dict:
    card = tomllib.loads((unit_dir / "card.toml").read_text())
    pq = sorted(unit_dir.glob("*.parquet"))[0]
    df = pd.read_parquet(pq)
    acol = "asset" if "asset" in df.columns else "asset_id"
    df["date"] = pd.to_datetime(df["date"].astype(str).str[:10])
    wide = df.pivot(index="date", columns=acol, values="value").sort_index().astype(float)
    t = card["targets"]
    return dict(unit=card["task"]["id"], panel=pq.stem, card=card, wide=wide,
                asof=pd.Timestamp(card["provenance"]["data_cutoff"]),
                assets=list(t["asset_ids"]), horizons=[int(h) for h in t["horizons"]],
                target_type=t["target_type"], value_unit=t["value_unit"],
                freq="monthly" if pq.stem == "macro_monthly" else "daily")

UNITS = [load_unit(p) for p in sorted(REPO.glob("units/t2-F1-*"))]
survey = pd.DataFrame([{
    "unit": u["unit"], "panel": u["panel"], "freq": u["freq"], "as-of": u["asof"].date(),
    "targets": ",".join(u["assets"]), "horizons_bd": u["horizons"], "target_type": u["target_type"],
    "panel_rows": len(u["wide"]), "panel_end": u["wide"].index[-1].date(), "n_series": u["wide"].shape[1],
} for u in UNITS])
survey

,unit,panel,freq,as-of,targets,horizons_bd,target_type,panel_rows,panel_end,n_series
0,t2-F1-ai-mom-2024,factors_daily,daily,2024-05-31,MOM,[127],log_return,6370,2024-05-31,6
1,t2-F1-aud-on-hold-2016,g10_fx_daily,daily,2016-11-01,AUD,"[126, 189]",level,4232,2016-11-01,10
2,t2-F1-cad-boc-2017,g10_fx_daily,daily,2017-07-12,CAD,"[126, 189]",level,4404,2017-07-12,10
3,t2-F1-chf-highly-valued-2021,g10_fx_daily,daily,2021-06-17,CHF,"[126, 189]",level,5384,2021-06-17,10
4,t2-F1-conflicting-texts-2024,rates_daily,daily,2024-06-12,UST_10Y,"[126, 189]",level,6116,2024-06-12,6
5,t2-F1-considerable-period-2003,rates_daily,daily,2003-08-12,UST_10Y,"[126, 189]",level,903,2003-08-12,6
6,t2-F1-conundrum-2005,rates_daily,daily,2005-02-18,UST_10Y,"[126, 189]",level,1283,2005-02-18,6
7,t2-F1-cpi-glidepath-2023,macro_monthly,monthly,2023-07-12,CPI_ALL,"[140, 160]",level,281,2023-05-01,6
8,t2-F1-dkk-peg-2019,g10_fx_daily,daily,2019-03-01,DKK,[129],level,4809,2019-03-01,10
9,t2-F1-eur-range-2017,g10_fx_daily,daily,2017-09-07,EUR,"[126, 189]",level,4444,2017-09-07,10


In [4]:
print(survey.groupby(["panel", "freq", "target_type"]).size().rename("units").to_frame())
print()
print("distinct horizon sets:", sorted({tuple(h) for h in survey.horizons_bd}))

                                   units
panel         freq    target_type       
factors_daily daily   log_return       2
g10_fx_daily  daily   level            6
macro_monthly monthly level            2
rates_daily   daily   level           13

distinct horizon sets: [(126, 189), (127,), (128,), (129,), (140, 160), (145, 165)]


## 2 — Definitions used to build the table

| term | meaning |
|---|---|
| **origin** $t$ | a date in the panel at which we pretend to stand; the feature vector is computed from data ≤ $t$ |
| **horizon** $h$ | business days ahead, from the card (`horizon_bd`) |
| **steps_ahead** $k$ | how many *rows* ahead the target is: $k=h$ on daily panels; on monthly panels, the number of months between the last panel month and the month of (as-of + $h$ BD) |
| **working series** $x$ | `level` target → the level itself. `log_return` target → the *cumulative log-return index* $\sum\log(1+r)$, so that $x_{t+k}-x_t$ is exactly the cumulative log return the card asks for |
| **anchor** | $x_t$ for level targets, 0 for log-return targets (the competition's reference point) |
| **target** | the value in the competition's unit: $x_{t+k}$ (level) or $x_{t+k}-x_t$ (log return) |
| **target_change** | `target − anchor` = $x_{t+k}-x_t$ in both cases — the natural modelling target, comparable across units |
| **split** | `train` if the target is observed inside the panel; `predict` if $t$ is the as-of date (the row you actually forecast) |

Rows whose target falls after the as-of but that are not the as-of row are dropped: they have no
label and are not needed.

**Monthly wrinkle.** `t2-F1-cpi-glidepath-2023` is as-of 2023-07-12 but its panel ends 2023-05
(publication lag); 140 BD after the as-of is late Jan 2024, so the target is the Jan-2024 index —
8 months after the last panel row. `steps_ahead` carries that 8.

In [5]:
def working_series(wide: pd.DataFrame, asset: str, target_type: str) -> pd.Series:
    """The series whose differences are the 'steps'.
    level      -> the level itself.
    log_return -> cumulative log-return index, so diff = log(1+r) and x[t+h]-x[t] = the target."""
    s = wide[asset].dropna()                  # the asset's own trading days (ragged panels)
    if target_type == "log_return":
        return np.log1p(s).cumsum()
    return s

## 3 — Features

All computed from the working series at or before the origin. **Window names are in business
days; on monthly panels they are converted to months (21 BD → 1 month)** so one schema fits both.

### 3.1 Per-series features (every row)

| column | definition | what it captures |
|---|---|---|
| `level`, `log_level` | $x_t$, $\log x_t$ (NaN for log-return units — the cumulative index has no meaningful level) | mean-reversion, zero-bound |
| `mom_21/63/126/252` | $x_t - x_{t-w}$ | trend over 1 / 3 / 6 / 12 months |
| `z_252` | $(x_t - \bar x_{252})/s_{252}$ | level vs its 1-year mean, in sd |
| `pos_252` | position of $x_t$ within its 1-year min–max range, 0–1 | near a high or a low? |
| `rv_21/63/252` | std of daily steps over the window | realised volatility |
| `ewma_vol` | RiskMetrics EWMA vol, λ = 0.94 | current vol |
| `vol_ratio` | `rv_21 / rv_252` | vol regime: elevated or calm |
| `skew_252` | skewness of daily steps over 1 year | asymmetry |
| `last_step` | $x_t - x_{t-1}$ | yesterday's move |

In [6]:
# window names are in business days; on monthly panels they are converted to months (21 BD = 1 M)
WINDOWS = {"21": 21, "63": 63, "126": 126, "252": 252}

def series_features(x: pd.Series, freq: str, is_level: bool) -> pd.DataFrame:
    scale = 21 if freq == "monthly" else 1
    w = {k: max(1, v // scale) for k, v in WINDOWS.items()}
    d = x.diff()
    F = pd.DataFrame(index=x.index)
    F["level"]     = x if is_level else np.nan
    F["log_level"] = np.log(x.clip(lower=1e-3)) if is_level else np.nan
    for k in ("21", "63", "126", "252"):
        F[f"mom_{k}"] = x - x.shift(w[k])
    F["z_252"]     = (x - x.rolling(w["252"]).mean()) / x.rolling(w["252"]).std()
    F["pos_252"]   = (x - x.rolling(w["252"]).min()) / (x.rolling(w["252"]).max() - x.rolling(w["252"]).min())
    for k in ("21", "63", "252"):
        F[f"rv_{k}"]  = d.rolling(w[k]).std()
    F["ewma_vol"]  = d.ewm(alpha=1 - 0.94 ** scale).std()
    F["vol_ratio"] = F["rv_21"] / F["rv_252"]
    F["skew_252"]  = d.rolling(w["252"]).skew()
    F["last_step"] = d
    return F

### 3.2 Panel-context features (prefix `ctx_`, NaN where the panel doesn't have them)

| panel | columns | meaning |
|---|---|---|
| `rates_daily` | `ctx_slope_10_2`, `ctx_slope_5_2`, `ctx_curv_2_5_10`, `ctx_ust2y`, `ctx_ust10y`, `ctx_slope_mom_63` | curve shape and its 3-month change |
| `g10_fx_daily` | `ctx_usd_mom_63`, `ctx_usd_rv_63` | broad USD strength: every pair oriented as units-per-USD, averaged |
| `factors_daily` | `ctx_mkt_mom_63`, `ctx_mkt_rv_63`, `ctx_mom_mom_63` | equity market trend / vol, momentum-factor trend |
| `macro_monthly` | `ctx_cpi_yoy`, `ctx_core_yoy`, `ctx_unrate`, `ctx_unrate_chg_3`, `ctx_nfp_3m` | inflation, labour market |

A model trained across panels should either use only the per-series block, or treat `ctx_` NaNs
as "not applicable" (tree models do this natively; linear models need the panel-specific subset).

In [6]:
USD_PER_CCY = {"AUD", "EUR", "GBP", "NZD"}   # quoted as USD per unit; the rest are units per USD

def context_features(wide: pd.DataFrame, panel: str, freq: str) -> pd.DataFrame:
    C = pd.DataFrame(index=wide.index)
    m63 = 63 if freq == "daily" else 3
    if panel == "rates_daily":
        C["ctx_slope_10_2"] = wide["UST_10Y"] - wide["UST_2Y"]
        C["ctx_slope_5_2"]  = wide["UST_5Y"] - wide["UST_2Y"]
        C["ctx_curv_2_5_10"] = 2 * wide["UST_5Y"] - wide["UST_2Y"] - wide["UST_10Y"]
        C["ctx_ust2y"]      = wide["UST_2Y"]
        C["ctx_ust10y"]     = wide["UST_10Y"]
        C["ctx_slope_mom_63"] = C["ctx_slope_10_2"].diff(m63)
    elif panel == "g10_fx_daily":
        # orient every pair as units-per-USD so a rise = USD strength, then average the 63d log change
        lg = np.log(wide)
        lg = lg.apply(lambda col: -col if col.name in USD_PER_CCY else col)
        usd = lg.diff(m63).mean(axis=1)
        C["ctx_usd_mom_63"] = usd
        C["ctx_usd_rv_63"]  = lg.diff().mean(axis=1).rolling(m63).std()
    elif panel == "factors_daily":
        mkt = np.log1p(wide["MKT"].dropna()); mom = np.log1p(wide["MOM"].dropna())
        C["ctx_mkt_mom_63"] = mkt.cumsum().diff(m63)
        C["ctx_mkt_rv_63"]  = mkt.rolling(m63).std()
        C["ctx_mom_mom_63"] = mom.cumsum().diff(m63)
    elif panel == "macro_monthly":
        C["ctx_cpi_yoy"]    = wide["CPI_ALL"].pct_change(12) * 100
        C["ctx_core_yoy"]   = wide["CPI_CORE"].pct_change(12) * 100
        C["ctx_unrate"]     = wide["UNRATE"]
        C["ctx_unrate_chg_3"] = wide["UNRATE"].diff(3)
        C["ctx_nfp_3m"]     = wide["NFP"].diff(3)
    return C

## 4 — Build the table

In [7]:
def steps_ahead_for(u: dict, h: int) -> int:
    """Daily: h rows. Monthly: months between the last panel month and the month of (asof + h BD)."""
    if u["freq"] == "daily":
        return h
    target_date = u["asof"] + pd.offsets.BDay(h)
    last = u["wide"].index[-1]
    return (target_date.year - last.year) * 12 + (target_date.month - last.month)

def build_unit_rows(u: dict) -> pd.DataFrame:
    wide, is_level = u["wide"], u["target_type"] == "level"
    ctx = context_features(wide, u["panel"], u["freq"])
    out = []
    for asset in u["assets"]:
        x = working_series(wide, asset, u["target_type"])
        F = series_features(x, u["freq"], is_level)
        n = len(x); idx = x.index
        for h in u["horizons"]:
            k = steps_ahead_for(u, h)
            anchor = x.to_numpy() if is_level else np.zeros(n)
            tgt = np.full(n, np.nan); tdate = np.full(n, np.datetime64("NaT", "ns"))
            tgt[:n - k] = x.to_numpy()[k:] if is_level else (x.to_numpy()[k:] - x.to_numpy()[:n - k])
            tdate[:n - k] = idx[k:].to_numpy()
            D = pd.DataFrame({
                "unit": u["unit"], "family": "F1", "panel": u["panel"], "freq": u["freq"],
                "asset": asset, "target_type": u["target_type"], "value_unit": u["value_unit"],
                "asof": u["asof"], "origin_date": idx, "origin_idx": np.arange(n),
                "horizon_bd": h, "steps_ahead": k, "target_date": tdate,
                "anchor": anchor, "target": tgt,
            }, index=idx)
            D["target_change"] = D["target"] - D["anchor"]
            D["is_asof"] = D["origin_date"] == idx[-1]
            D["split"] = np.where(D["target"].notna(), "train", np.where(D["is_asof"], "predict", "drop"))
            D = pd.concat([D, F, ctx.reindex(idx).ffill()], axis=1)   # context aligned to the asset's days
            out.append(D[D["split"] != "drop"])
    return pd.concat(out, ignore_index=True)

def build_all(units_glob="units/t2-F1-*") -> pd.DataFrame:
    parts = [build_unit_rows(load_unit(p)) for p in sorted(REPO.glob(units_glob))]
    Xy = pd.concat(parts, ignore_index=True)
    # the as-of row of a monthly unit has target_date = asof + h BD (expected), not a panel date
    m = Xy["is_asof"] & Xy["target_date"].isna()
    Xy.loc[m, "target_date"] = Xy.loc[m, "asof"] + Xy.loc[m, "horizon_bd"].map(lambda h: pd.offsets.BDay(int(h)))
    return Xy

ID_COLS = ["unit", "family", "panel", "freq", "asset", "target_type", "value_unit", "asof",
           "origin_date", "origin_idx", "horizon_bd", "steps_ahead", "target_date", "is_asof", "split"]
TARGET_COLS = ["anchor", "target", "target_change"]


ID_COLS = ["unit", "family", "panel", "freq", "asset", "target_type", "value_unit", "asof",
           "origin_date", "origin_idx", "horizon_bd", "steps_ahead", "target_date", "is_asof", "split"]
TARGET_COLS = ["anchor", "target", "target_change"]

t0 = time.time()
Xy = pd.concat([build_unit_rows(u) for u in UNITS], ignore_index=True)
# the as-of row of a monthly unit has no panel date to point at: use the expected as-of + h BD
m = Xy["is_asof"] & Xy["target_date"].isna()
Xy.loc[m, "target_date"] = [a + pd.offsets.BDay(int(h)) for a, h in zip(Xy.loc[m, "asof"], Xy.loc[m, "horizon_bd"])]
FEATURE_COLS = [c for c in Xy.columns if c not in ID_COLS + TARGET_COLS]
print(f"{len(Xy):,} rows × {Xy.shape[1]} columns in {time.time()-t0:.1f}s")
print(f"{len(ID_COLS)} id columns, {len(TARGET_COLS)} target columns, {len(FEATURE_COLS)} feature columns")

150,066 rows × 49 columns in 0.2s
15 id columns, 3 target columns, 31 feature columns


### 4.1 — Schema

In [8]:
schema = pd.DataFrame({"dtype": Xy.dtypes.astype(str),
                       "role": ["id" if c in ID_COLS else "target" if c in TARGET_COLS else "feature" for c in Xy.columns],
                       "non-null %": (Xy.notna().mean()*100).round(1),
                       "example": Xy.loc[Xy.is_asof & (Xy.unit == "t2-F1-hawkish-cut-2024") & (Xy.horizon_bd == 126)].iloc[0].values})
schema

,dtype,role,non-null %,example
unit,str,id,100.0,t2-F1-hawkish-cut-2024
family,str,id,100.0,F1
panel,str,id,100.0,rates_daily
freq,str,id,100.0,daily
asset,str,id,100.0,UST_2Y
target_type,str,id,100.0,level
value_unit,str,id,100.0,percent_per_annum
asof,datetime64[us],id,100.0,2024-12-18 00:00:00
origin_date,datetime64[us],id,100.0,2024-12-18 00:00:00
origin_idx,int64,id,100.0,6244


### 4.2 — Row counts

In [9]:
print(Xy.groupby(["freq", "target_type", "split"]).size().rename("rows").to_frame(), "\n")
per_unit = Xy.groupby(["unit", "asset"]).agg(horizons=("horizon_bd", lambda s: sorted(set(s))),
                                            train_rows=("split", lambda s: (s == "train").sum()),
                                            predict_rows=("split", lambda s: (s == "predict").sum()),
                                            first_origin=("origin_date", "min"), asof=("asof", "first"))
per_unit

                               rows
freq    target_type split          
daily   level       predict      37
                    train    138135
        log_return  predict       2
                    train     10772
monthly level       predict       4
                    train      1116 



,,horizons,train_rows,predict_rows,first_origin,asof
unit,asset,,,,,
t2-F1-ai-mom-2024,MOM,[127],6015,1,2000-01-03,2024-05-31
t2-F1-aud-on-hold-2016,AUD,"[126, 189]",8149,2,2000-01-03,2016-11-01
t2-F1-cad-boc-2017,CAD,"[126, 189]",8493,2,2000-01-03,2017-07-12
t2-F1-chf-highly-valued-2021,CHF,"[126, 189]",10453,2,2000-01-03,2021-06-17
t2-F1-conflicting-texts-2024,UST_10Y,"[126, 189]",11917,2,2000-01-03,2024-06-12
t2-F1-considerable-period-2003,UST_10Y,"[126, 189]",1491,2,2000-01-03,2003-08-12
t2-F1-conundrum-2005,UST_10Y,"[126, 189]",2251,2,2000-01-03,2005-02-18
t2-F1-cpi-glidepath-2023,CPI_ALL,"[140, 160]",545,2,2000-01-01,2023-07-12
t2-F1-dkk-peg-2019,DKK,[129],4680,1,2000-01-03,2019-03-01


## 5 — Sanity checks

**(a) The rows to forecast.** Exactly one `predict` row per (unit, asset, horizon) — 43 in total —
each sitting on the unit's last panel date with the anchor equal to the last observed value.

In [17]:
pred = Xy[Xy.split == "predict"]
assert pred.groupby(["unit", "asset", "horizon_bd"]).size().eq(1).all()
assert (pred.origin_date == pred.groupby("unit").origin_date.transform("max")).all()
pred[["unit", "asset", "target_type", "horizon_bd", "steps_ahead", "origin_date", "target_date", "anchor",
      "level", "mom_63", "rv_63", "ctx_slope_10_2", "ctx_usd_mom_63", "ctx_mkt_mom_63", "ctx_cpi_yoy"]].pipe(lambda d: d.round({c: 4 for c in d.select_dtypes("number").columns}))

,unit,asset,target_type,horizon_bd,steps_ahead,origin_date,target_date,anchor,level,mom_63,rv_63,ctx_slope_10_2,ctx_usd_mom_63,ctx_mkt_mom_63,ctx_cpi_yoy
6015,t2-F1-ai-mom-2024,MOM,log_return,127,127,2024-05-31,2024-11-26,0.0000,NaN,-0.0029,0.0081,NaN,NaN,0.0152,NaN
10122,t2-F1-aud-on-hold-2016,AUD,level,126,126,2016-11-01,2017-04-26,0.7659,0.7659,0.0048,0.0040,NaN,0.0218,NaN,NaN
14166,t2-F1-aud-on-hold-2016,AUD,level,189,189,2016-11-01,2017-07-24,0.7659,0.7659,0.0048,0.0040,NaN,0.0218,NaN,NaN
18445,t2-F1-cad-boc-2017,CAD,level,126,126,2017-07-12,2018-01-04,1.2768,1.2768,-0.0534,0.0059,NaN,-0.0402,NaN,NaN
22661,t2-F1-cad-boc-2017,CAD,level,189,189,2017-07-12,2018-04-03,1.2768,1.2768,-0.0534,0.0059,NaN,-0.0402,NaN,NaN
27920,t2-F1-chf-highly-valued-2021,CHF,level,126,126,2021-06-17,2021-12-10,0.9173,0.9173,-0.0128,0.0041,NaN,0.0030,NaN,NaN
33116,t2-F1-chf-highly-valued-2021,CHF,level,189,189,2021-06-17,2022-03-09,0.9173,0.9173,-0.0128,0.0041,NaN,0.0030,NaN,NaN
39107,t2-F1-conflicting-texts-2024,UST_10Y,level,126,126,2024-06-12,2024-12-05,4.3100,4.3100,0.1200,0.0610,-0.44,NaN,NaN,NaN
45035,t2-F1-conflicting-texts-2024,UST_10Y,level,189,189,2024-06-12,2025-03-04,4.3100,4.3100,0.1200,0.0610,-0.44,NaN,NaN,NaN
45813,t2-F1-considerable-period-2003,UST_10Y,level,126,126,2003-08-12,2004-02-04,4.3700,4.3700,0.7400,0.0762,2.66,NaN,NaN,NaN


**(b) The target is what the card asks for.** Reconstruct a few training targets directly from the
raw panel and compare.

In [18]:
# level target: the 2Y on 2019-10-30 + 126 BD, read straight from the panel
u = next(u for u in UNITS if u["unit"] == "t2-F1-hawkish-cut-2024")
s = u["wide"]["UST_2Y"]; i = s.index.get_loc(pd.Timestamp("2019-10-30"))
row = Xy[(Xy.unit == u["unit"]) & (Xy.origin_date == "2019-10-30") & (Xy.horizon_bd == 126)].iloc[0]
print(f"level  : panel says {s.iloc[i+126]:.2f} on {s.index[i+126].date()}   table says target={row.target:.2f} target_date={row.target_date.date()}")
assert np.isclose(row.target, s.iloc[i+126]) and row.target_date == s.index[i+126]

# log-return target: sum of log(1+r) over the 127 days after the origin, from the raw panel
u = next(u for u in UNITS if u["unit"] == "t2-F1-ai-mom-2024")
r = u["wide"]["MOM"].dropna(); j = r.index.get_loc(pd.Timestamp("2020-03-02"))
direct = np.log1p(r.iloc[j+1:j+1+127]).sum()
row = Xy[(Xy.unit == u["unit"]) & (Xy.origin_date == "2020-03-02")].iloc[0]
print(f"logret : direct sum {direct:+.4f}   table says target={row.target:+.4f} anchor={row.anchor} target_change={row.target_change:+.4f}")
assert np.isclose(row.target, direct)

# the repo's own step definition agrees
from qfbench2_track_forecasting.targets import log_return_steps
steps = pd.Series(log_return_steps(r), index=r.index)
assert np.isclose(steps.iloc[j+1:j+1+127].sum(), direct)
print("matches qfbench2_track_forecasting.targets.log_return_steps ✓")

level  : panel says 0.19 on 2020-05-04   table says target=0.19 target_date=2020-05-04
logret : direct sum -0.0730   table says target=-0.0730 anchor=0.0 target_change=-0.0730
matches qfbench2_track_forecasting.targets.log_return_steps ✓


matches qfbench2_track_forecasting.targets.log_return_steps ✓


**(c) No look-ahead in the features.** Every feature at origin $t$ must be unchanged if the panel is
truncated at $t$. Rebuild one unit with its panel cut early and compare the overlapping rows.

In [19]:
u_full = next(u for u in UNITS if u["unit"] == "t2-F1-hawkish-cut-2024")
u_cut = dict(u_full); u_cut["wide"] = u_full["wide"].loc[:"2015-06-30"]; u_cut["asof"] = pd.Timestamp("2015-06-30")
a = build_unit_rows(u_full).set_index(["origin_date", "horizon_bd"])
b = build_unit_rows(u_cut).set_index(["origin_date", "horizon_bd"])
common = b.index[b.index.get_level_values(0) <= "2015-06-30"]
fc = [c for c in FEATURE_COLS if c in a.columns]           # this unit's own feature columns
diff = (a.loc[common, fc] - b.loc[common, fc]).abs().max().max()
print(f"max |feature difference| between full panel and panel truncated at 2015-06-30: {diff:.2e}")
assert diff < 1e-9

max |feature difference| between full panel and panel truncated at 2015-06-30: 0.00e+00


**(d) Where the NaNs are.** Warm-up windows at the start of each series, `level` on log-return
units, and `ctx_` columns on panels that don't have them. Nothing else.

In [20]:
nan = Xy[FEATURE_COLS].isna().mean().mul(100).round(1).rename("NaN %").to_frame()
nan["reason"] = np.where(nan.index.str.startswith("ctx_"), "other panels", np.where(nan.index.isin(["level", "log_level"]), "log-return units + warm-up", "warm-up window"))
nan.T

,level,log_level,mom_21,mom_63,mom_126,mom_252,z_252,pos_252,rv_21,rv_63,rv_252,ewma_vol,vol_ratio,skew_252,last_step,ctx_mkt_mom_63,ctx_mkt_rv_63,ctx_mom_mom_63,ctx_usd_mom_63,ctx_usd_rv_63,ctx_slope_10_2,ctx_slope_5_2,ctx_curv_2_5_10,ctx_ust2y,ctx_ust10y,ctx_slope_mom_63,ctx_cpi_yoy,ctx_core_yoy,ctx_unrate,ctx_unrate_chg_3,ctx_nfp_3m
NaN %,7.2,7.2,0.5,1.6,3.3,6.6,6.6,6.6,1.3,1.6,6.6,0.1,7.3,6.6,0.0,92.9,92.9,92.9,67.6,67.6,40.8,40.8,40.8,40.8,40.8,41.9,99.3,99.3,99.3,99.3,99.3
reason,log-return units + warm-up,log-return units + warm-up,warm-up window,warm-up window,warm-up window,warm-up window,warm-up window,warm-up window,warm-up window,warm-up window,warm-up window,warm-up window,warm-up window,warm-up window,warm-up window,other panels,other panels,other panels,other panels,other panels,other panels,other panels,other panels,other panels,other panels,other panels,other panels,other panels,other panels,other panels,other panels


## 6 — One unit up close

The last few training rows and the predict row for the notebook's earlier unit, so the shape of a
row is concrete.

In [21]:
cols = ["origin_date", "horizon_bd", "split", "anchor", "target", "target_change", "level", "mom_63", "mom_252", "z_252", "rv_63", "ewma_vol", "ctx_slope_10_2"]
one = Xy[(Xy.unit == "t2-F1-hawkish-cut-2024") & (Xy.horizon_bd == 126)]
pd.concat([one[one.split == "train"].tail(3), one[one.split == "predict"]])[cols].pipe(lambda d: d.round({c: 4 for c in d.select_dtypes("number").columns}))

,origin_date,horizon_bd,split,anchor,target,target_change,level,mom_63,mom_252,z_252,rv_63,ewma_vol,ctx_slope_10_2
85194,2024-06-13,126,train,4.68,4.25,-0.43,4.68,0.00,0.13,-0.3379,0.0606,0.0590,-0.44
85195,2024-06-14,126,train,4.67,4.25,-0.42,4.67,-0.05,0.00,-0.3797,0.0604,0.0572,-0.47
85196,2024-06-17,126,train,4.75,4.35,-0.40,4.75,0.02,0.01,-0.0455,0.0612,0.0595,-0.47
85197,2024-12-18,126,predict,4.35,NaN,NaN,4.35,0.76,-0.02,-0.0602,0.0552,0.0516,0.15


## 7 — Save

In [22]:
Xy.to_parquet(OUT / "f1_Xy.parquet", index=False)
Xy[Xy.split == "predict"].to_parquet(OUT / "f1_Xy_predict.parquet", index=False)
(OUT / "f1_columns.json").write_text(json.dumps({"id": ID_COLS, "target": TARGET_COLS, "feature": FEATURE_COLS}, indent=2))
for p in sorted(OUT.glob("f1_*")):
    print(f"{p.name:<24} {p.stat().st_size/1e6:6.2f} MB")

f1_Xy.parquet             11.11 MB
f1_Xy_predict.parquet      0.03 MB
f1_columns.json            0.00 MB


In [23]:
Xy

,unit,family,panel,freq,asset,target_type,value_unit,asof,origin_date,origin_idx,horizon_bd,steps_ahead,target_date,anchor,target,target_change,is_asof,split,level,log_level,mom_21,mom_63,mom_126,mom_252,z_252,pos_252,rv_21,rv_63,rv_252,ewma_vol,vol_ratio,skew_252,last_step,ctx_mkt_mom_63,ctx_mkt_rv_63,ctx_mom_mom_63,ctx_usd_mom_63,ctx_usd_rv_63,ctx_slope_10_2,ctx_slope_5_2,ctx_curv_2_5_10,ctx_ust2y,ctx_ust10y,ctx_slope_mom_63,ctx_cpi_yoy,ctx_core_yoy,ctx_unrate,ctx_unrate_chg_3,ctx_nfp_3m
0,t2-F1-ai-mom-2024,F1,factors_daily,daily,MOM,log_return,cumulative_log_return,2024-05-31,2000-01-03,0,127,127,2000-07-05,0.00,0.065924,0.065924,False,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,t2-F1-ai-mom-2024,F1,factors_daily,daily,MOM,log_return,cumulative_log_return,2024-05-31,2000-01-04,1,127,127,2000-07-06,0.00,0.091486,0.091486,False,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.019183,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,t2-F1-ai-mom-2024,F1,factors_daily,daily,MOM,log_return,cumulative_log_return,2024-05-31,2000-01-05,2,127,127,2000-07-07,0.00,0.107338,0.107338,False,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.010091,NaN,NaN,-0.004912,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,t2-F1-ai-mom-2024,F1,factors_daily,daily,MOM,log_return,cumulative_log_return,2024-05-31,2000-01-06,3,127,127,2000-07-10,0.00,0.119144,0.119144,False,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.007253,NaN,NaN,-0.014911,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,t2-F1-ai-mom-2024,F1,factors_daily,daily,MOM,log_return,cumulative_log_return,2024-05-31,2000-01-07,4,127,127,2000-07-11,0.00,0.117054,0.117054,False,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.011262,NaN,NaN,0.005783,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150061,t2-F1-ust-positioning-2018,F1,rates_daily,daily,UST_10Y,level,percent_per_annum,2018-09-28,2017-12-22,4498,189,189,2018-09-25,2.48,3.100000,0.620000,False,train,2.48,0.908259,0.16,0.22,0.33,-0.07,1.323470,0.754386,0.029984,0.029901,0.035441,0.030452,0.846018,-0.112244,0.000000,NaN,NaN,NaN,NaN,NaN,0.57,0.35,0.13,1.91,2.48,-0.23,NaN,NaN,NaN,NaN,NaN
150062,t2-F1-ust-positioning-2018,F1,rates_daily,daily,UST_10Y,level,percent_per_annum,2018-09-28,2017-12-26,4499,189,189,2018-09-26,2.47,3.060000,0.590000,False,train,2.47,0.904218,0.13,0.25,0.33,-0.08,1.242311,0.736842,0.030079,0.029432,0.035447,0.029777,0.848576,-0.108891,-0.010000,NaN,NaN,NaN,NaN,NaN,0.55,0.33,0.11,1.92,2.47,-0.23,NaN,NaN,NaN,NaN,NaN
150063,t2-F1-ust-positioning-2018,F1,rates_daily,daily,UST_10Y,level,percent_per_annum,2018-09-28,2017-12-27,4500,189,189,2018-09-27,2.42,3.060000,0.640000,False,train,2.42,0.883768,0.10,0.18,0.21,-0.13,0.801687,0.649123,0.032034,0.030130,0.035585,0.031781,0.900226,-0.101699,-0.050000,NaN,NaN,NaN,NaN,NaN,0.53,0.33,0.13,1.89,2.42,-0.26,NaN,NaN,NaN,NaN,NaN
150064,t2-F1-ust-positioning-2018,F1,rates_daily,daily,UST_10Y,level,percent_per_annum,2018-09-28,2017-12-28,4501,189,189,2018-09-28,2.43,3.050000,0.620000,False,train,2.43,0.887891,0.09,0.12,0.21,-0.14,0.904424,0.666667,0.031870,0.028897,0.035567,0.030877,0.896057,-0.099141,0.010000,NaN,NaN,NaN,NaN,NaN,0.52,0.32,0.12,1.91,2.43,-0.32,NaN,NaN,NaN,NaN,NaN


In [25]:
Xy[Xy.unit == "t2-F1-hawkish-cut-2024"]

,unit,family,panel,freq,asset,target_type,value_unit,asof,origin_date,origin_idx,horizon_bd,steps_ahead,target_date,anchor,target,target_change,is_asof,split,level,log_level,mom_21,mom_63,mom_126,mom_252,z_252,pos_252,rv_21,rv_63,rv_252,ewma_vol,vol_ratio,skew_252,last_step,ctx_mkt_mom_63,ctx_mkt_rv_63,ctx_mom_mom_63,ctx_usd_mom_63,ctx_usd_rv_63,ctx_slope_10_2,ctx_slope_5_2,ctx_curv_2_5_10,ctx_ust2y,ctx_ust10y,ctx_slope_mom_63,ctx_cpi_yoy,ctx_core_yoy,ctx_unrate,ctx_unrate_chg_3,ctx_nfp_3m
79078,t2-F1-hawkish-cut-2024,F1,rates_daily,daily,UST_2Y,level,percent_per_annum,2024-12-18,2000-01-03,0,126,126,2000-07-03,6.38,6.31,-0.07,False,train,6.38,1.853168,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.20,0.12,0.04,6.38,6.58,NaN,NaN,NaN,NaN,NaN,NaN
79079,t2-F1-hawkish-cut-2024,F1,rates_daily,daily,UST_2Y,level,percent_per_annum,2024-12-18,2000-01-04,1,126,126,2000-07-05,6.30,6.29,-0.01,False,train,6.30,1.840550,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.08,NaN,NaN,NaN,NaN,NaN,0.19,0.10,0.01,6.30,6.49,NaN,NaN,NaN,NaN,NaN,NaN
79080,t2-F1-hawkish-cut-2024,F1,rates_daily,daily,UST_2Y,level,percent_per_annum,2024-12-18,2000-01-05,2,126,126,2000-07-06,6.38,6.34,-0.04,False,train,6.38,1.853168,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.113137,NaN,NaN,0.08,NaN,NaN,NaN,NaN,NaN,0.24,0.13,0.02,6.38,6.62,NaN,NaN,NaN,NaN,NaN,NaN
79081,t2-F1-hawkish-cut-2024,F1,rates_daily,daily,UST_2Y,level,percent_per_annum,2024-12-18,2000-01-06,3,126,126,2000-07-07,6.35,6.29,-0.06,False,train,6.35,1.848455,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.081021,NaN,NaN,-0.03,NaN,NaN,NaN,NaN,NaN,0.22,0.11,0.00,6.35,6.57,NaN,NaN,NaN,NaN,NaN,NaN
79082,t2-F1-hawkish-cut-2024,F1,rates_daily,daily,UST_2Y,level,percent_per_annum,2024-12-18,2000-01-07,4,126,126,2000-07-10,6.31,6.31,0.00,False,train,6.31,1.842136,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.067039,NaN,NaN,-0.04,NaN,NaN,NaN,NaN,NaN,0.21,0.11,0.01,6.31,6.52,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91250,t2-F1-hawkish-cut-2024,F1,rates_daily,daily,UST_2Y,level,percent_per_annum,2024-12-18,2024-03-13,6052,189,189,2024-12-13,4.61,4.25,-0.36,False,train,4.61,1.528228,0.15,-0.10,-0.36,0.58,0.071630,0.597222,0.065433,0.073005,0.085126,0.062577,0.768654,-0.489869,0.03,NaN,NaN,NaN,NaN,NaN,-0.42,-0.42,-0.42,4.61,4.19,0.06,NaN,NaN,NaN,NaN,NaN
91251,t2-F1-hawkish-cut-2024,F1,rates_daily,daily,UST_2Y,level,percent_per_annum,2024-12-18,2024-03-14,6053,189,189,2024-12-16,4.68,4.25,-0.43,False,train,4.68,1.543298,0.04,-0.05,-0.30,0.48,0.252881,0.645833,0.054371,0.073513,0.084573,0.062675,0.642889,-0.514612,0.07,NaN,NaN,NaN,NaN,NaN,-0.39,-0.39,-0.39,4.68,4.29,0.14,NaN,NaN,NaN,NaN,NaN
91252,t2-F1-hawkish-cut-2024,F1,rates_daily,daily,UST_2Y,level,percent_per_annum,2024-12-18,2024-03-15,6054,189,189,2024-12-17,4.72,4.25,-0.47,False,train,4.72,1.551809,0.16,0.26,-0.24,0.79,0.352991,0.673611,0.051566,0.065096,0.082839,0.061232,0.622487,-0.450112,0.04,NaN,NaN,NaN,NaN,NaN,-0.41,-0.39,-0.37,4.72,4.31,0.01,NaN,NaN,NaN,NaN,NaN
91253,t2-F1-hawkish-cut-2024,F1,rates_daily,daily,UST_2Y,level,percent_per_annum,2024-12-18,2024-03-18,6055,189,189,2024-12-18,4.73,4.35,-0.38,False,train,4.73,1.553925,0.17,0.36,-0.27,0.59,0.374454,0.680556,0.051538,0.063974,0.081800,0.059367,0.630049,-0.503077,0.01,NaN,NaN,NaN,NaN,NaN,-0.39,-0.37,-0.35,4.73,4.34,0.06,NaN,NaN,NaN,NaN,NaN


## 8 — How to use it (read before modelling)

```python
Xy = pd.read_parquet("data_prep/f1_Xy.parquet")
cols = json.load(open("data_prep/f1_columns.json"))
X, y = Xy[cols["feature"]], Xy["target_change"]      # model the change, add the anchor back at the end
```

Three things that will bite if ignored:

**1. Leakage across units.** Units on the same panel share history — the 2Y series in
`t2-F1-hawkish-cut-2024` contains every earlier rates unit's target. If you train one pooled model
and use it to forecast unit *u*, restrict the training rows to those whose outcome was knowable at
*u*'s as-of:

```python
train_u = Xy[(Xy.split == "train") & (Xy.target_date <= asof_u)]
```

This is also the rule for a fair backtest: origin $t$ may only be trained on rows with
`target_date <= t`. (For the sealed final set, whose as-of dates are later than everything here,
the whole table is legal training data.)

**2. Duplicated rows.** The same (panel, asset, origin, horizon) appears in every unit that shares
that panel and asset — e.g. the 2Y at h=126 appears in 7 units. For pooled training, dedupe:

```python
Xy.drop_duplicates(["panel", "asset", "origin_date", "horizon_bd"])
```

**3. Overlapping targets.** Consecutive daily origins share most of their 126-day outcome window,
so 6,000 rows are worth ~30 independent observations per horizon. Regularise hard, validate with
*time-blocked* folds (never random K-fold), and consider subsampling origins with
`origin_idx % 21 == 0` for a cleaner training set.

Horizons are a column, not separate tables — either train one model per `horizon_bd` (or
`steps_ahead`) or feed it as a feature. `freq == "monthly"` rows have windows in months; filter
them out for a daily-only model.